# 01 - Data Preparation

Download Stockfish NNUE training data, convert to our plaintext format,
encode as Board768 features, and save preprocessed `.npz` arrays to Google Drive.

**Data source**: [linrock/test80-2024](https://huggingface.co/datasets/linrock/test80-2024) binpack files,
rescored with Stockfish + 7-piece Syzygy tablebases.

**Output**: `kanue/data/{train,val,test}.npz` on Google Drive.

---

## 0. Mount Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/kanue'
!mkdir -p {DRIVE_BASE}/data {DRIVE_BASE}/checkpoints {DRIVE_BASE}/logs {DRIVE_BASE}/results

In [ ]:
# Install kanue package from the repo
!pip install -q git+https://github.com/y0sif/kanue.git
!pip install -q python-chess tqdm

## 1. Download Training Data

We use the Stockfish nnue-pytorch toolchain to convert `.binpack` to plaintext.

**Option A**: Download a pre-converted plaintext dataset (faster).

**Option B**: Download `.binpack` and convert using the nnue-pytorch tools.

We'll start with Option A using a smaller dataset for initial experiments,
then scale up once we've validated the approach.

In [ ]:
import os
from pathlib import Path

DATA_DIR = Path(DRIVE_BASE) / 'data'
RAW_DIR = Path('/content/raw_data')
RAW_DIR.mkdir(exist_ok=True)

# Check if we already have preprocessed data
if (DATA_DIR / 'train.npz').exists():
    print('Preprocessed data already exists on Drive. Skip to Section 4.')
else:
    print('No preprocessed data found. Proceeding with download...')

### Option A: Build from Stockfish self-play (recommended)

We'll use Stockfish to generate training positions with evaluations.
This gives us full control over data quality and volume.

In [ ]:
# Download Stockfish binary for data generation
!wget -q https://github.com/official-stockfish/Stockfish/releases/latest/download/stockfish-ubuntu-x86-64-avx2.tar -O /tmp/sf.tar
!tar xf /tmp/sf.tar -C /tmp/
!find /tmp/stockfish* -name 'stockfish' -type f | head -1 | xargs -I{} cp {} /usr/local/bin/stockfish
!chmod +x /usr/local/bin/stockfish
!stockfish <<< 'uci' | head -5

In [ ]:
import subprocess
import chess
import chess.engine
import random
from tqdm import tqdm


def generate_positions(
    n_games: int = 5000,
    depth: int = 8,
    output_path: str = '/content/raw_data/positions.txt',
    random_plies: int = 8,
):
    """Generate training positions via Stockfish self-play.

    For each game:
    1. Play `random_plies` random legal moves to diversify openings
    2. Then play with Stockfish, recording each position + eval + result
    """
    engine = chess.engine.SimpleEngine.popen_uci('/usr/local/bin/stockfish')
    engine.configure({'Threads': 2, 'Hash': 256})

    positions = []

    for game_idx in tqdm(range(n_games), desc='Generating games'):
        board = chess.Board()

        # Random opening moves for diversity
        for _ in range(random_plies):
            legal = list(board.legal_moves)
            if not legal:
                break
            board.push(random.choice(legal))

        game_positions = []

        # Self-play with Stockfish
        while not board.is_game_over() and board.fullmove_number < 200:
            result = engine.analyse(board, chess.engine.Limit(depth=depth))
            score = result['score'].white()

            if score.is_mate():
                # Skip mate scores (not useful for eval training)
                cp = 10000 if score.mate() > 0 else -10000
            else:
                cp = score.score()

            # Record position (skip extreme evals and checks)
            if abs(cp) <= 3000 and not board.is_check():
                game_positions.append((board.fen(), cp))

            # Play the best move
            best_move = result.get('pv', [None])[0]
            if best_move is None:
                break
            board.push(best_move)

        # Determine game result as WDL
        outcome = board.outcome()
        if outcome is None:
            wdl = 0.5  # Draw (game ended by move limit)
        elif outcome.winner == chess.WHITE:
            wdl = 1.0
        elif outcome.winner == chess.BLACK:
            wdl = 0.0
        else:
            wdl = 0.5

        # Write positions with game result
        for fen, cp in game_positions:
            positions.append(f'{fen} | {cp} | {wdl}')

    engine.quit()

    # Shuffle and write
    random.shuffle(positions)
    with open(output_path, 'w') as f:
        f.write('\n'.join(positions))

    print(f'Generated {len(positions)} positions from {n_games} games')
    return output_path

In [ ]:
# Generate positions -- adjust n_games based on your time budget
# ~5000 games at depth 8 takes ~30-60 min on Colab
# This produces ~500k-1M training positions
PLAINTEXT_PATH = str(RAW_DIR / 'positions.txt')

if not os.path.exists(PLAINTEXT_PATH):
    generate_positions(n_games=5000, depth=8, output_path=PLAINTEXT_PATH)
else:
    print(f'Positions file already exists: {PLAINTEXT_PATH}')

# Count lines
!wc -l {PLAINTEXT_PATH}

## 2. Convert to Board768 Features

In [ ]:
from kanue.data import load_positions_from_plaintext

print('Loading and encoding positions...')
stm, nstm, targets = load_positions_from_plaintext(
    PLAINTEXT_PATH,
    max_positions=None,  # Load all
    max_cp=3000,
    wdl_weight=0.0,  # Pure eval prediction for now
)

print(f'Loaded {len(targets)} positions')
print(f'STM features shape: {stm.shape}')
print(f'Target range: [{targets.min():.4f}, {targets.max():.4f}]')
print(f'Target mean: {targets.mean():.4f} (should be ~0.5 for balanced data)')

## 3. Train/Val/Test Split

In [ ]:
import numpy as np

n = len(targets)
indices = np.random.permutation(n)

# 80/10/10 split
train_end = int(0.8 * n)
val_end = int(0.9 * n)

train_idx = indices[:train_end]
val_idx = indices[train_end:val_end]
test_idx = indices[val_end:]

print(f'Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}')

## 4. Save to Google Drive

In [ ]:
# Save preprocessed arrays to Drive
for split_name, idx in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    path = DATA_DIR / f'{split_name}.npz'
    np.savez_compressed(
        path,
        stm=stm[idx],
        nstm=nstm[idx],
        targets=targets[idx],
    )
    size_mb = path.stat().st_size / 1024 / 1024
    print(f'Saved {split_name}: {len(idx)} positions ({size_mb:.1f} MB) -> {path}')

# Also save the raw plaintext as backup
import shutil
shutil.copy(PLAINTEXT_PATH, DATA_DIR / 'positions.txt')
print(f'\nAll data saved to {DATA_DIR}')

## 5. Quick Sanity Check

In [ ]:
import matplotlib.pyplot as plt

# Verify we can reload from Drive
train_data = np.load(DATA_DIR / 'train.npz')
print(f"Reloaded train set: {train_data['stm'].shape}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Target distribution
axes[0].hist(train_data['targets'].flatten(), bins=50, alpha=0.7)
axes[0].set_xlabel('Target (win probability)')
axes[0].set_ylabel('Count')
axes[0].set_title('Target Distribution')

# Feature density (avg active features per position)
active_per_pos = train_data['stm'].sum(axis=1)
axes[1].hist(active_per_pos, bins=30, alpha=0.7)
axes[1].set_xlabel('Active features per position')
axes[1].set_ylabel('Count')
axes[1].set_title('Feature Density (should be ~16-32)')

plt.tight_layout()
plt.savefig(str(DATA_DIR.parent / 'results' / 'data_distribution.png'), dpi=150)
plt.show()

print('\nData preparation complete. Proceed to notebook 02.')